# ReAct Prompting with LangChain Agents

**Course:** Natural Language Processing · Unit 4 — Prompt Engineering  
**Institution:** Universidad Tecnológica de Bolívar  
**Reference:** Jurafsky, D. & Martin, J. H. (2025). *Speech and Language Processing* (3rd ed.), Ch. 12 — Prompting and In-Context Learning  
**Python compatibility:** 3.10 · 3.11 · 3.12 · 3.13

---

## Learning Objectives

1. Understand the ReAct (Reason + Act) prompting paradigm
2. Configure a LangChain agent with OpenAI as the backbone LLM
3. Wire up external tools (SerpAPI for web search, LLM-Math for calculations)
4. Execute a multi-step reasoning query and observe the Thought/Action/Observation loop


---

## 1. Install Dependencies

Run **once** in your terminal before opening the notebook:

```bash
pip install openai langchain langchain-community python-dotenv google-search-results
```


In [ ]:
# pip install openai langchain langchain-community python-dotenv google-search-results
# Run the line above once in your terminal, then restart the kernel.
import os

import openai
from langchain.agents import AgentType, initialize_agent, load_tools
from langchain.llms import OpenAI


---

## 2. Configure API Credentials

Store secrets as environment variables — never hard-code them in a notebook.

```bash
export OPENAI_API_KEY=sk-...
export SERPAPI_API_KEY=your-serpapi-key
```


In [ ]:
# Read credentials from environment — set them before launching Jupyter
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "your-openai-api-key-here")
SERPAPI_API_KEY = os.environ.get("SERPAPI_API_KEY", "your-serpapi-api-key-here")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["SERPAPI_API_KEY"] = SERPAPI_API_KEY


---

## 3. Initialise the Language Model


In [ ]:
# temperature=0 makes the model deterministic — best for structured reasoning loops
llm = OpenAI(
    temperature=0,
    openai_api_key=os.getenv("OPENAI_API_KEY"),
)


---

## 4. Load External Tools

Two tools are wired up:
- **`serpapi`** — live Google Search (requires a SerpAPI key)
- **`llm-math`** — arithmetic calculation handled by the LLM

If you do not have a SerpAPI key, comment out `'serpapi'` and use only `'llm-math'`.


In [ ]:
tools = load_tools(
    ["serpapi", "llm-math"],
    llm=llm,
)


---

## 5. Initialise the ReAct Agent

`ZERO_SHOT_REACT_DESCRIPTION` uses the ReAct prompting format: the agent alternates  
between **Thought** (reasoning), **Action** (tool call), and **Observation** (result)  
until it reaches a **Final Answer**.


In [ ]:
agent = initialize_agent(
    tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
)


---

## 6. Execute a Multi-Step Query

This question requires two distinct capabilities:
1. **Web search** — to find the person's current age
2. **Math** — to raise that age to the 0.23 power


In [ ]:
question = "Who is Olivia Wilde's boyfriend? What is his current age raised to the 0.23 power?"
response = agent.run(question)

print("\nFinal Answer:", response)


> **Output interpretation:** With verbose=True, each Thought/Action/Observation step is printed. The agent first searches for Olivia Wilde's partner, extracts the age, then calls the llm-math tool to compute age^0.23. If the search tool is unavailable, the agent may fail at the Action step — a reminder that tool reliability is a critical dependency in ReAct systems.


---

## Summary

ReAct (Reason + Act) interleaves language model reasoning with tool calls:

```
Thought  → plan the next step
Action   → call a tool (search / calculator / code executor)
Observation → receive tool output
... repeat until Final Answer
```

This paradigm allows LLMs to handle tasks that require up-to-date information  
or precise computation — capabilities that exceed the model's parametric knowledge.
